# 05 pi_0 权限 smoke 与训练门控

        这一节不急着启动长训练，而是先检查 PaliGemma、Hugging Face gated 权限、权重下载、数据加载和 1-step smoke。只有 smoke 通过后，正式训练才有意义。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：权限检查命令模板


In [ ]:
print("""
cd "$PROJECT_ROOT"
HF_TOKEN_STDIN=1 REQUIRE_PROXY=1 ./install_hf_token_for_pi0.sh
""")


这个命令形态强调两件事：token 不写进 Notebook，脚本先验证 `whoami`、PaliGemma 和 `lerobot/pi0`，全部通过后再保存。


## Checkpoint 2：1-step smoke 证明什么


In [ ]:
rows = [
    ("gated model 权限", "能读取 PaliGemma / pi0 所需配置"),
    ("数据加载", "能读取 LeRobot 数据集和 observation/action key"),
    ("policy 构造", "pi0 policy 能初始化"),
    ("一次训练步", "forward/backward/optimizer step 能跑通"),
    ("checkpoint 写出", "输出目录出现 smoke checkpoint"),
]
md_table(["检查项", "通过后说明什么"], rows)


1-step smoke 不证明策略收敛，也不代表成功率。它只是正式训练前的门槛。


## Checkpoint 3：训练命令模板


In [ ]:
print("""
cd "$PROJECT_ROOT"
RUN_SMOKE=1 RUN_FULL_TRAIN=1 PI0_STEPS=20000 PI0_BATCH_SIZE=4 \\
  ./run_pi0_train_eval_after_hf_ready.sh
""")


正式训练建议放到终端、tmux 或后台脚本中执行。Notebook 负责记录参数、解释门控和读取训练后的 summary。


## Checkpoint 4：阶段性状态图


In [ ]:
show_asset("model_status_summary.png", width=900)
